# M4 · Visión como *input* — launcher de Agustín

Este notebook **solo orquesta** el módulo `M4/vision/`. No reimplementa la visión, el harness ni la app.

- **Qué hace:** imagen de una nota clínica → transcripción → sistema de M3.
- **Qué NO hace:** eval set con imágenes (Luis/Pau), harness y delta (Isabella), app (Pau).
- **Demo:** usa una nota clínica **sintética** y marcada. No subas fotos de pacientes al repo.

Ver `M4/vision/README.md` para el alcance completo.


In [ ]:
# En Colab: descomentar y ajustar la URL/rama si hace falta
# !git clone https://github.com/luisNP21/Topicos-IA.git /content/Topicos-IA
# %cd /content/Topicos-IA
# !pip install -r M4/vision/requirements.txt

import json
from pathlib import Path
print("cwd:", Path.cwd())


## 1 · Humo con backend falso (sin GPU, sin Docling, sin API)

Valida el cableado: manifiesto → transcripción → cache → resumen.


In [ ]:
!python M4/vision/run_vision.py --config M4/vision/config.yaml --demo --backend fake


In [ ]:
cache = Path("M4/vision/predictions/transcripciones_fake.json")
print(json.dumps(json.loads(cache.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)[:1200])


## 2 · Corridas reales (elegir una)

- `--backend vlm`: Qwen2-VL-2B-Instruct (GPU T4).
- `--backend ocr_docling`: Docling (CPU; baseline recomendado por Isabella).
- `--backend ocr_tesseract`: requiere `tesseract-ocr` + `tesseract-ocr-spa`.
- `--comparar`: corre VLM y OCR sobre los mismos casos y deja un cache por backend.
- `--ner`: además escribe `M2/predictions/vision_<backend>.json` para el harness de M2.


In [ ]:
# Demo sintética con VLM (GPU T4)
# !python M4/vision/run_vision.py --config M4/vision/config.yaml --demo --backend vlm

# Demo sintética con Docling (CPU)
# !python M4/vision/run_vision.py --config M4/vision/config.yaml --demo --backend ocr_docling

# Eval set del equipo (cuando exista) + comparación + cache NER
# !python M4/vision/run_vision.py --config M4/vision/config.yaml --manifest M4/eval_set/casos_imagen.csv --comparar --ner


## 3 · Fronteras del reparto

| Pieza | Dueño |
|---|---|
| Módulo `vision/` (VLM + OCR, `run`, cache) | **Agustín** |
| Justificación del patrón | Pau |
| Set sintético degradado | Luis |
| Casos límite y fotos reales | Pau |
| Harness y delta M3 vs. M4 | Isabella |
| CER/WER y auditoría de alucinaciones | Luis (apoyo de Agustín: `juez.py`) |
| App Gradio/Streamlit | Pau |

No se reportan métricas inventadas: los números reales salen de correr los backends sobre el eval set del equipo.
